# Sundays of Advent

In [1]:
import pycantus
import pycantus.data as data
from pycantus.filtration import Filter
import copy
import utils
import importlib

In [2]:
corpus = data.load_dataset('cantuscorpus_v1.0')

Loading chants and sources...
Data loaded!


In [3]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 888010
Number of sources in CantusCorpus v1.0: 2278


In [4]:
# Clean the corpus
# Drop doxology
doxo_filter = pycantus.filtration.Filter('doxo_filter')
doxo_filter.add_value_exclude('cantus_id', '909000')
corpus.apply_filter(doxo_filter)

# Drop fragments => sources with less than 100 chants
corpus.drop_small_sources_data(min_chants=100)

In [5]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 855146
Number of sources in CantusCorpus v1.0: 510


In [6]:
sigla_dict = {source.srclink: source.siglum for source in corpus.sources}

### Complete days

In [7]:
# Build corpora for Sundays, e.g. "Dom. 1 Adventus"
advent_sundays_corpora = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(corpus)
    sunday_filter = Filter(f'{feast}_filter')
    sunday_filter.add_value_include('feast', feast)
    sunday_corpus.apply_filter(sunday_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_corpora[feast] = sunday_corpus
    print(f'{feast}: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus: 6447 chants, 245 sources
Dom. 2 Adventus: 4544 chants, 240 sources
Dom. 3 Adventus: 4876 chants, 251 sources
Dom. 4 Adventus: 4689 chants, 251 sources


In [8]:
# Bulit copora for advent sundays Lauds only
advent_sundays_lauds_corpora = {}
lauds_filter = Filter(f'lauds_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(advent_sundays_corpora[feast])
    lauds_filter.add_value_include('office', 'L')
    sunday_corpus.apply_filter(lauds_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_lauds_corpora[feast] = sunday_corpus
    print(f'{feast} Lauds: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus Lauds: 833 chants, 102 sources
Dom. 2 Adventus Lauds: 648 chants, 97 sources
Dom. 3 Adventus Lauds: 689 chants, 106 sources
Dom. 4 Adventus Lauds: 727 chants, 112 sources


In [9]:
# Corpora for vespers only
advent_sundays_vespers_corpora = {}
vespers_filter = Filter(f'vespers_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(advent_sundays_corpora[feast])
    vespers_filter.add_value_include('office', 'V')
    sunday_corpus.apply_filter(vespers_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_vespers_corpora[feast] = sunday_corpus
    print(f'{feast} Vespers: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus Vespers: 489 chants, 111 sources
Dom. 2 Adventus Vespers: 240 chants, 91 sources
Dom. 3 Adventus Vespers: 294 chants, 99 sources
Dom. 4 Adventus Vespers: 225 chants, 99 sources


In [10]:
# corpora for second vespers only
advent_sundays_second_vespers_corpora = {}
second_vespers_filter = Filter(f'second_vespers_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(advent_sundays_corpora[feast])
    second_vespers_filter.add_value_include('office', 'V2')
    sunday_corpus.apply_filter(second_vespers_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_second_vespers_corpora[feast] = sunday_corpus
    print(f'{feast} Second Vespers: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus Second Vespers: 349 chants, 95 sources
Dom. 2 Adventus Second Vespers: 236 chants, 91 sources
Dom. 3 Adventus Second Vespers: 231 chants, 93 sources
Dom. 4 Adventus Second Vespers: 166 chants, 79 sources


In [11]:
# Corpora for matins only
advent_sundays_matins_corpora = {}
matins_filter = Filter(f'matins_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(advent_sundays_corpora[feast])
    matins_filter.add_value_include('office', 'M')
    sunday_corpus.apply_filter(matins_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_matins_corpora[feast] = sunday_corpus
    print(f'{feast} Matins: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus Matins: 2871 chants, 117 sources
Dom. 2 Adventus Matins: 2082 chants, 108 sources
Dom. 3 Adventus Matins: 2349 chants, 118 sources
Dom. 4 Adventus Matins: 2311 chants, 122 sources


### Networks

In [12]:
import networkx as nx

In [13]:
advent_sundays_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_network.edgelist')

Dom. 1 Adventus: 245 nodes, 13735 edges
Dom. 2 Adventus: 240 nodes, 13306 edges
Dom. 3 Adventus: 251 nodes, 13927 edges
Dom. 4 Adventus: 251 nodes, 14066 edges


In [14]:
advent_sundays_lauds_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_lauds_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_lauds_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_lauds_network.edgelist')

Dom. 1 Adventus: 102 nodes, 4728 edges
Dom. 2 Adventus: 97 nodes, 4441 edges
Dom. 3 Adventus: 106 nodes, 4983 edges
Dom. 4 Adventus: 112 nodes, 5566 edges


In [15]:
advent_sundays_vespers_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_vespers_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_vespers_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_vespers_network.edgelist')

Dom. 1 Adventus: 111 nodes, 3976 edges
Dom. 2 Adventus: 91 nodes, 1079 edges
Dom. 3 Adventus: 99 nodes, 2269 edges
Dom. 4 Adventus: 99 nodes, 1083 edges


In [16]:
advent_sundays_vespers2_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_second_vespers_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_vespers2_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_second_vespers_network.edgelist')

Dom. 1 Adventus: 95 nodes, 3117 edges
Dom. 2 Adventus: 91 nodes, 2213 edges
Dom. 3 Adventus: 93 nodes, 1603 edges
Dom. 4 Adventus: 79 nodes, 422 edges


In [17]:
advent_sundays_matins_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_matins_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_matins_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_matins_network.edgelist()')

Dom. 1 Adventus: 117 nodes, 5228 edges
Dom. 2 Adventus: 108 nodes, 4782 edges
Dom. 3 Adventus: 118 nodes, 5368 edges
Dom. 4 Adventus: 122 nodes, 6216 edges


In [18]:
for G in advent_sundays_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 245 (3)
       Edges | 13,735 (0)
      Degree | 112.12 (227)


  Components | 97.6% (5)
  Clustering | 0.9426
Modularity from Louvain | 0.4286 (6)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 240 (0)
       Edges | 13,306 (0)
      Degree | 110.88 (230)
  Components | 98.8% (2)
  Clustering | 0.9752
Modularity from Louvain | 0.4121 (4)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 251 (1)
       Edges | 13,927 (0)
      Degree | 110.97 (233)
  Components | 97.2% (4)
  Clustering | 0.9603
Modularity from Louvain | 0.4415 (7)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 251 (0)
       Edges | 14,066 (0)
      Degree | 112.08 (237)
  Components | 98.4% (2)
  Clustering | 0.9652
Modularity from Louvain | 0.4952 (4)

--------------------------------------


In [19]:
for G in advent_sundays_lauds_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 102 (0)
       Edges | 4,728 (0)
      Degree | 92.71 (100)
  Components | 100.0% (1)
  Clustering | 0.9672
Modularity from Louvain | 0.0306 (2)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 97 (0)
       Edges | 4,441 (0)
      Degree | 91.57 (96)
  Components | 100.0% (1)
  Clustering | 0.9691
Modularity from Louvain | 0.0399 (3)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 106 (1)
       Edges | 4,983 (0)
      Degree | 94.02 (103)
  Components | 99.1% (2)
  Clustering | 0.9586
Modularity from Louvain | 0.0452 (4)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 112 (0)
       Edges | 5,566 (0)
      Degree | 99.39 (109)
  Components | 100.0% (1)
  Clustering | 0.9640
Modularity from Louvain | 0.0436 (2)

--------------------------------------


In [20]:
for G in advent_sundays_vespers_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 111 (3)
       Edges | 3,976 (0)
      Degree | 71.64 (99)
  Components | 95.5% (5)
  Clustering | 0.8682
Modularity from Louvain | 0.1437 (8)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 91 (5)
       Edges | 1,079 (0)
      Degree | 23.71 (51)
  Components | 94.5% (6)
  Clustering | 0.7108
Modularity from Louvain | 0.3498 (8)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 99 (3)
       Edges | 2,269 (0)
      Degree | 45.84 (78)
  Components | 94.9% (5)
  Clustering | 0.8404
Modularity from Louvain | 0.2058 (8)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 99 (10)
       Edges | 1,083 (0)
      Degree | 21.88 (61)
  Components | 89.9% (11)
  Clustering | 0.7135
Modularity from Louvain | 0.3157 (16)

--------------------------------------


In [21]:
for G in advent_sundays_matins_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 117 (4)
       Edges | 5,228 (0)
      Degree | 89.37 (108)
  Components | 94.9% (6)
  Clustering | 0.8898
Modularity from Louvain | 0.1026 (8)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 108 (1)
       Edges | 4,782 (0)
      Degree | 88.56 (103)
  Components | 99.1% (2)
  Clustering | 0.9581
Modularity from Louvain | 0.1106 (5)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 118 (0)
       Edges | 5,368 (0)
      Degree | 90.98 (109)
  Components | 98.3% (2)
  Clustering | 0.9416
Modularity from Louvain | 0.1047 (7)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 122 (0)
       Edges | 6,216 (0)
      Degree | 101.90 (117)
  Components | 100.0% (1)
  Clustering | 0.9690
Modularity from Louvain | 0.0893 (5)

--------------------------------------


In [22]:
for G in advent_sundays_vespers2_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 95 (3)
       Edges | 3,117 (0)
      Degree | 65.62 (85)
  Components | 96.8% (4)
  Clustering | 0.8916
Modularity from Louvain | 0.1294 (7)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 91 (4)
       Edges | 2,213 (0)
      Degree | 48.64 (74)
  Components | 95.6% (5)
  Clustering | 0.8710
Modularity from Louvain | 0.1501 (8)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 93 (3)
       Edges | 1,603 (0)
      Degree | 34.47 (68)
  Components | 94.6% (5)
  Clustering | 0.8105
Modularity from Louvain | 0.3343 (9)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 79 (6)
       Edges | 422 (0)
      Degree | 10.68 (31)
  Components | 92.4% (7)
  Clustering | 0.7799
Modularity from Louvain | 0.5952 (14)

--------------------------------------


In [23]:
THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

In [24]:
import pandas as pd
import itertools
importlib.reload(utils)

def compare_edgewise_networks(networks, thresholds):
    rows = []
    for feast1, feast2 in itertools.combinations(networks.keys(), 2):
        row = {'network_1': feast1, 'network_2': feast2}
        for threshold in thresholds:
            result = utils.compare_edge_overlap(networks[feast1], networks[feast2], threshold=threshold)
            row['edges_1'] = result['edges_1']
            row['edges_2'] = result['edges_2']
            row['shared_edges'] = result['shared_edges']
            row['edge_overlap_' + str(threshold)] = result[f'edge_overlap_{threshold}']
        rows.append(row)
    overlap_df = pd.DataFrame(rows)
    return overlap_df

In [25]:
comparison_df = compare_edgewise_networks(advent_sundays_nets, THRESHOLDS)
comparison_df

,network_1,network_2,edges_1,edges_2,shared_edges,edge_overlap_0.2,edge_overlap_0.3,edge_overlap_0.4,edge_overlap_0.5,edge_overlap_0.6,edge_overlap_0.7,edge_overlap_0.8
0,Dom. 1 Adventus,Dom. 2 Adventus,1258,2083,877,0.853163,0.771816,0.702353,0.675325,0.604634,0.543627,0.355925
1,Dom. 1 Adventus,Dom. 3 Adventus,1258,1212,579,0.783312,0.725827,0.654233,0.619407,0.555982,0.447522,0.306187
2,Dom. 1 Adventus,Dom. 4 Adventus,1258,323,142,0.606528,0.490960,0.353516,0.257748,0.173689,0.113889,0.098680
3,Dom. 2 Adventus,Dom. 3 Adventus,2083,1212,791,0.826305,0.782548,0.746851,0.725545,0.632285,0.472041,0.315895
4,Dom. 2 Adventus,Dom. 4 Adventus,2083,323,179,0.647063,0.514477,0.378695,0.278614,0.163162,0.107012,0.080377
5,Dom. 3 Adventus,Dom. 4 Adventus,1212,323,189,0.688988,0.560330,0.391753,0.272102,0.179020,0.129611,0.140416


In [26]:
comparison_df = compare_edgewise_networks(advent_sundays_lauds_nets, THRESHOLDS)
comparison_df

,network_1,network_2,edges_1,edges_2,shared_edges,edge_overlap_0.2,edge_overlap_0.3,edge_overlap_0.4,edge_overlap_0.5,edge_overlap_0.6,edge_overlap_0.7,edge_overlap_0.8
0,Dom. 1 Adventus,Dom. 2 Adventus,349,639,79,0.825321,0.808855,0.734296,0.590897,0.380079,0.195469,0.086909
1,Dom. 1 Adventus,Dom. 3 Adventus,349,1123,110,0.778007,0.776208,0.708946,0.613270,0.392610,0.194733,0.080764
2,Dom. 1 Adventus,Dom. 4 Adventus,349,772,84,0.710100,0.706037,0.644440,0.546407,0.348416,0.192143,0.081003
3,Dom. 2 Adventus,Dom. 3 Adventus,639,1123,304,0.779259,0.770918,0.723843,0.618099,0.448527,0.299406,0.208505
4,Dom. 2 Adventus,Dom. 4 Adventus,639,772,168,0.737095,0.723850,0.665478,0.549040,0.357956,0.273444,0.135157
5,Dom. 3 Adventus,Dom. 4 Adventus,1123,772,205,0.821985,0.823753,0.781776,0.698622,0.469332,0.303404,0.121302


In [27]:
comparison_df = compare_edgewise_networks(advent_sundays_vespers_nets, THRESHOLDS)
comparison_df

,network_1,network_2,edges_1,edges_2,shared_edges,edge_overlap_0.2,edge_overlap_0.3,edge_overlap_0.4,edge_overlap_0.5,edge_overlap_0.6,edge_overlap_0.7,edge_overlap_0.8
0,Dom. 1 Adventus,Dom. 2 Adventus,34,33,0,0.168309,0.098546,0.060383,0.043197,0.016043,0.010526,0.000000
1,Dom. 1 Adventus,Dom. 3 Adventus,34,44,0,0.237787,0.125490,0.075179,0.043178,0.024038,0.008850,0.000000
2,Dom. 1 Adventus,Dom. 4 Adventus,34,35,0,0.119731,0.072430,0.033479,0.026804,0.005650,0.010989,0.000000
3,Dom. 2 Adventus,Dom. 3 Adventus,33,44,1,0.198825,0.119555,0.094059,0.078390,0.098684,0.062500,0.013158
4,Dom. 2 Adventus,Dom. 4 Adventus,33,35,1,0.130132,0.092405,0.067873,0.066138,0.031250,0.012658,0.014925
5,Dom. 3 Adventus,Dom. 4 Adventus,44,35,0,0.177641,0.111615,0.068740,0.056452,0.033333,0.000000,0.000000


In [28]:
comparison_df = compare_edgewise_networks(advent_sundays_matins_nets, THRESHOLDS)
comparison_df

,network_1,network_2,edges_1,edges_2,shared_edges,edge_overlap_0.2,edge_overlap_0.3,edge_overlap_0.4,edge_overlap_0.5,edge_overlap_0.6,edge_overlap_0.7,edge_overlap_0.8
0,Dom. 1 Adventus,Dom. 2 Adventus,34,62,12,0.775490,0.582682,0.399796,0.350154,0.337662,0.313514,0.142857
1,Dom. 1 Adventus,Dom. 3 Adventus,34,63,11,0.685788,0.520000,0.348584,0.290893,0.275210,0.239631,0.127907
2,Dom. 1 Adventus,Dom. 4 Adventus,34,93,16,0.606168,0.403061,0.316367,0.311978,0.281377,0.254386,0.144144
3,Dom. 2 Adventus,Dom. 3 Adventus,62,63,26,0.782561,0.638585,0.460120,0.412434,0.380165,0.359649,0.262626
4,Dom. 2 Adventus,Dom. 4 Adventus,62,93,33,0.711280,0.509006,0.404924,0.404484,0.390782,0.334694,0.270492
5,Dom. 3 Adventus,Dom. 4 Adventus,63,93,34,0.814765,0.596536,0.423525,0.404586,0.442202,0.357692,0.278689


In [29]:
comparison_df = compare_edgewise_networks(advent_sundays_vespers2_nets, THRESHOLDS)
comparison_df

,network_1,network_2,edges_1,edges_2,shared_edges,edge_overlap_0.2,edge_overlap_0.3,edge_overlap_0.4,edge_overlap_0.5,edge_overlap_0.6,edge_overlap_0.7,edge_overlap_0.8
0,Dom. 1 Adventus,Dom. 2 Adventus,39,195,4,0.332854,0.194055,0.129118,0.099894,0.054878,0.016260,0.017391
1,Dom. 1 Adventus,Dom. 3 Adventus,39,92,2,0.227685,0.120080,0.077011,0.070444,0.059829,0.013605,0.015504
2,Dom. 1 Adventus,Dom. 4 Adventus,39,40,0,0.070334,0.031145,0.017621,0.014374,0.024242,0.021053,0.000000
3,Dom. 2 Adventus,Dom. 3 Adventus,195,92,10,0.393130,0.236860,0.146996,0.133333,0.058462,0.035842,0.036101
4,Dom. 2 Adventus,Dom. 4 Adventus,195,40,0,0.068894,0.030349,0.022305,0.019582,0.015326,0.000000,0.000000
5,Dom. 3 Adventus,Dom. 4 Adventus,92,40,0,0.070898,0.049711,0.031315,0.032483,0.012121,0.000000,0.000000


In [52]:
import sbmodel 
importlib.reload(sbmodel)
importlib.reload(utils)

def perform_weighted_sbm_analysis(network):
    model = sbmodel.SBModel()
    model.load_graph_nx(network)
    model.fit_sbm_weighted('weight')
    best_state = model.best_states['Weighted_DC_SBM']
    partitions = utils.get_partitions_from_state(best_state, sigla_dict=sigla_dict)
    return partitions

In [55]:
def prepare_sbm_by_pairs(nets):
    sbm_results_by_pairs = {}
    for feast1, feast2 in itertools.combinations(nets.keys(), 2):
        network1, network2 = utils.networks_reduction_on_shared_nodes(advent_sundays_nets[feast1], advent_sundays_nets[feast2])
        partition1 = perform_weighted_sbm_analysis(network1)[1]
        partition2 = perform_weighted_sbm_analysis(network2)[1]
        sbm_results_by_pairs[(feast1, feast2)] = (partition1, partition2)
        
    return sbm_results_by_pairs

In [56]:
advent_sunday_sbm_results = prepare_sbm_by_pairs(advent_sundays_nets)

Loaded graph with 236 vertices, 13350 edges
Fitting weighted SBM to graph with 236 vertices and 13350 edges...
Fitting weighted SBM (init 1/10)...
[1/10] entropy = 14634.10
Fitting weighted SBM (init 2/10)...
[2/10] entropy = 14656.29
Fitting weighted SBM (init 3/10)...
[3/10] entropy = 14629.15
Fitting weighted SBM (init 4/10)...
[4/10] entropy = 14635.21
Fitting weighted SBM (init 5/10)...
[5/10] entropy = 14639.74
Fitting weighted SBM (init 6/10)...
[6/10] entropy = 14643.47
Fitting weighted SBM (init 7/10)...
[7/10] entropy = 14702.60
Fitting weighted SBM (init 8/10)...
[8/10] entropy = 14636.13
Fitting weighted SBM (init 9/10)...
[9/10] entropy = 14629.36
Fitting weighted SBM (init 10/10)...
[10/10] entropy = 15200.38
Loaded graph with 236 vertices, 12931 edges
Fitting weighted SBM to graph with 236 vertices and 12931 edges...
Fitting weighted SBM (init 1/10)...
[1/10] entropy = 15221.25
Fitting weighted SBM (init 2/10)...
[2/10] entropy = 15295.78
Fitting weighted SBM (init 3/10)

In [72]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def transform_partition_to_labels(partition):
    node_to_block = {}
    for block, nodes in partition.items():
        for node in nodes:
            node_to_block[node] = block
    return node_to_block


def compare_weighted_sbm_partitions(sbm_on_nets):
    rows = []
    for (feast1, feast2), (partition1, partition2) in sbm_on_nets.items():
        row = {'network_1': feast1, 'network_2': feast2}
        row['num_partitions_1'] = len(partition1)
        row['num_partitions_2'] = len(partition2)

        partition1 = transform_partition_to_labels(partition1)
        partition2 = transform_partition_to_labels(partition2)

        shared_nodes = sorted(set(partition1) & set(partition2))
        labels1 = [partition1[node] for node in shared_nodes]
        labels2 = [partition2[node] for node in shared_nodes]

        row['n_shared_nodes'] = len(shared_nodes)
        row['adjusted_rand_index'] = adjusted_rand_score(labels1, labels2)
        row['normalized_mutual_info'] = normalized_mutual_info_score(labels1, labels2)

        rows.append(row)
    comparison_df = pd.DataFrame(rows)
    return comparison_df

In [73]:
sbm_comparison_df = compare_weighted_sbm_partitions(advent_sunday_sbm_results)
sbm_comparison_df

,network_1,network_2,num_partitions_1,num_partitions_2,n_shared_nodes,adjusted_rand_index,normalized_mutual_info
0,Dom. 1 Adventus,Dom. 2 Adventus,9,10,236,0.681469,0.664744
1,Dom. 1 Adventus,Dom. 3 Adventus,9,8,238,0.713180,0.660525
2,Dom. 1 Adventus,Dom. 4 Adventus,8,7,225,0.659429,0.643655
3,Dom. 2 Adventus,Dom. 3 Adventus,11,7,238,0.684467,0.699504
4,Dom. 2 Adventus,Dom. 4 Adventus,10,7,228,0.675853,0.679746
5,Dom. 3 Adventus,Dom. 4 Adventus,8,6,236,0.827679,0.770018


## Summary accross Sundays

In [74]:
edge_comparison_df = compare_edgewise_networks(advent_sundays_nets, THRESHOLDS)
sbm_comparison_df = compare_weighted_sbm_partitions(advent_sunday_sbm_results)
advent_sunday_comparison_df = edge_comparison_df.merge(sbm_comparison_df, on=['network_1', 'network_2'])
advent_sunday_comparison_df.to_csv('visual/advent_sunday_comparison.csv', index=False)

## Summary accross Office (V, L, M, V2)